In [4]:
import math
#Using a scalar class that represents the node in each graph for any kind of mathematical equation
#The whole point of the autograd is that each sort of individual operation has its own unique way of taking the derivative
#For example: if a = b + c, then da/db = 1 or da/dc = 1. However, if a = b*c, then da/db = c and da/dc = b. Each of these operations
#Have a different way of computing their local derivatives. In a bigger more convoluted equation like a = b*c + d/f - c+b,
#The equation can be reconstrcuted by the correct order of operations and the chain rule can be applied starting from the local derivative
#all the way up to the desired result.


class Scalar:
    #Each scalar has a set of children its DIRECTLY derived from. NOT the indirect. The indirect children 
    #can be obtained via traversing backwards in the graph
    def __init__(self, digit, children:set = (), operation = ""):
        self.digit = digit
        self.children = children
        self.operation = operation
        self.back = lambda: None
        #By default, the gradient is taken of a final expression that will be specified
        self.gradient = 0
    
    #Building out the basic operations.
    def __add__(self, other:Scalar):
        res = Scalar(self.digit + other.digit, {self, other}, "add")
        def back():
            #The local derivative of an added expression w.r.t 1 element (ex: d(a + b + c)/da = 1) is 1
            #res.gradient is the derivative of the FINAL output with respect to this current res
            self.gradient += res.gradient
            #Same logic applies for other
            other.gradient += res.gradient
        res.back = back
        return res
    
    def __mul__(self, other:Scalar):
        res = Scalar(self.digit * other.digit, {self, other}, "mul")
        def back():
            #Local derivative (d(ab)/da = b) and (d(ab)/db = a) 
            #Multiply local with global derivative
            self.gradient += other.digit * res.gradient
            other.gradient += self.digit * res.gradient
        res.back = back
        return res
    
    #Using true div since we want floats
    def __truediv__(self, other:Scalar):
        res = Scalar(self.digit / other.digit, {self, other}, "div")

        def back():
            #Local derivative (d(a/b)/da = 1/b) and (d(a/b)/db = -a*b^-2)
            #Multiply with global derivative
            self.gradient += res.gradient * (1/other.digit)
            other.gradient += self.digit * (-(other.digit**-2)) * res.gradient

        res.back = back
        return res
    
    def __sub__(self, other:Scalar):
        res = Scalar(self.digit - other.digit, {self, other}, "sub")
        def back():
            #Local derivative (d(a-b)/da = 1) and (d(a-b)/db = -1)
            #Multiply with global derivative
            self.gradient += res.gradient
            other.gradient += -res.gradient
        res.back = back
        return res
    #arctan
    def tanh(self):
        res = Scalar(math.tanh(self.digit), {self}, "tanh")
        def back():
            #Local derivative (d(tanh(a) = 1 - tanh^2(a)) 
            #Multiply with global derivative
            self.gradient = (1 - (math.tanh(self.digit))**2) * res.gradient
        
        res.back = back

        return res


    

In [5]:
a = Scalar(2.0)
b = Scalar(3.0)
c = Scalar(1.0)

f = (a * b) + (a - c)

print(f"f = {f.digit}")  # should be 7.0

f.gradient = 1.0

f.back()

# f = mul_res + sub_res
# need to call back on intermediate nodes too
# for now manually:
for child in f.children:
    child.back()

print(f"a.gradient = {a.gradient}")  # should be 4.0
print(f"b.gradient = {b.gradient}")  # should be 2.0
print(f"c.gradient = {c.gradient}")  # should be -1.0

f = 7.0
a.gradient = 4.0
b.gradient = 2.0
c.gradient = -1.0
